## Exercise

Build a prediction script on the first 64 examples of `wikiann` (`en`) using:
- BERT NER model: `dslim/bert-base-NER`
- Llama 1B chat model (for entity extraction via prompt)

The output must be a CSV file with columns:
- `id`
- `bert_prediction`
- `llama_prediction`

Below you have a draft solution scaffold: some pieces are complete, while key parts are left as TODOs for you to implement.

In [1]:
import os, json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

print('Transformers version:', __import__('transformers').__version__)
print('Torch version:', torch.__version__)

BASE_DIR = os.path.join('lab2')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'models_cache')
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

Transformers version: 5.0.0
Torch version: 2.10.0+cu128


### Exercise: Fine-tune BERT on WikiANN

> Goal: train the token-classification model on the WikiANN train split and evaluate on validation.

Fill in the blanks in the next cell. Focus on data preprocessing, label alignment, and training args.

In [2]:
# Exercise scaffold: fill in all TODOs.
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
 )

# -----------------------------
# 1) Load WikiANN
# -----------------------------
wikiann = load_dataset('wikiann', 'en')
N_TRAIN_EXAMPLES = 10000 # TODO: select a subset of the training set for faster experimentation
wikiann['train'] = wikiann['train'].select(range(N_TRAIN_EXAMPLES))
label_list = wikiann['train'].features['ner_tags'].feature.names
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

# -----------------------------
# 2) Tokenizer + label alignment
# -----------------------------
model_checkpoint = 'dslim/bert-base-NER'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        truncation=True,
    )
    # TODO: Align the word-level labels to token-level labels.
    # - Use word_ids() to map tokens back to word indices.
    # - Use -100 for special tokens so they are ignored in loss.
    # - For subword tokens, keep the same label or convert I- tags as needed.
    aligned_labels = []
    # TODO: implement label alignment
    for i, labels in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # special tokens
            if word_idx is None:
                label_ids.append(-100)
            # first subword of a word
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])

            # subsequent subwords of the same word
            else:
                label_id = labels[word_idx]
                label_name = id_to_label[label_id]

                # if the original word was tagged as "B-" convert trailing subwords to "I-",
                # the trailing subwords must be converted to "I-" (Inside)
                if label_name.startswith("B-"):
                    updated_label_name = "I-" + label_name[2:]
                    label_ids.append(label_to_id[updated_label_name])
                else:
                    # if it was "O" or already "I-", keep same label
                    label_ids.append(label_id)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)


    tokenized['labels'] = aligned_labels
    return tokenized

tokenized = wikiann.map(tokenize_and_align_labels, batched=True)

# -----------------------------
# 3) Model
# -----------------------------
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id,
    ignore_mismatched_sizes=True
 )

# -----------------------------
# 4) Training setup
# -----------------------------
training_args = TrainingArguments(
    output_dir='lab4/models_cache/wikiann_bert_ner',
    eval_strategy='steps',  # TODO: set to 'epoch' or 'steps' or set eval_strategy
    learning_rate=3e-5,        # TODO: e.g., 2e-5
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.1,
 )

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=data_collator,
 )

# -----------------------------
# 5) Train + evaluate
# -----------------------------
# TODO: run training and evaluation
trainer.train()
metrics = trainer.evaluate()
print(metrics)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.31863588094711304, 'eval_runtime': 23.3998, 'eval_samples_per_second': 427.355, 'eval_steps_per_second': 26.71, 'epoch': 3.0}


In [3]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=4c975d1a323513a79247ed33352904f2db268ca8031c9b17fed2f2ea56f73e3a
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [10]:
# Draft solution scaffold (students complete TODO sections)
from datasets import load_dataset
from transformers import pipeline
import csv

# -----------------------------
# 6) Configuration
# -----------------------------
BERT_MODEL_ID = 'lab4/models_cache/wikiann_bert_ner/checkpoint-1875' # Chose whether to use the model you trained, or to load an already trained model like 'dslim/bert-base-NER'
LLAMA_MODEL_ID = 'unsloth/Llama-3.2-1B-Instruct'  # TODO: replace with the 1B Llama available to you
MAX_SAMPLES = 64
OUT_CSV = os.path.join(DATA_DIR, 'exercise_predictions.csv')

# -----------------------------
# 7) Load data
# -----------------------------
wikiann_eval = load_dataset('wikiann', 'en', split=f'validation[:{MAX_SAMPLES}]')
print('Loaded examples:', len(wikiann_eval))

# -----------------------------
# 8) Load models/tokenizers
# -----------------------------
# TODO: Load BERT tokenizer/model for token classification
bert_tok = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
bert_model = AutoModelForTokenClassification.from_pretrained(BERT_MODEL_ID)

# TODO: Load Llama tokenizer/model for generation/chat
llama_tok = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID)
llama_model = AutoModelForCausalLM.from_pretrained(LLAMA_MODEL_ID, torch_dtype=torch.float16, device_map='auto')

# -----------------------------
# 9) Prediction helpers
# -----------------------------

def predict_bert_ner(text):
    """Return a compact string summary of BERT entities for one text."""
    # # TODO: implement with the BERT token classification model.
    # # Suggested output format: "PERSON: Barack Obama | ORG: Stanford University | LOC: California"
    inputs = bert_tok(text, return_tensors='pt', return_offsets_mapping=True, truncation=True)
    offsets = inputs.pop("offset_mapping")[0].tolist()
    inputs = {k: v.to(bert_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = bert_model(**inputs).logits[0]

    pred_ids = logits.argmax(dim=-1).tolist()
    id2label = bert_model.config.id2label

    entities = []
    curr_ent = None

    # iterate through tokens and reconstruct continuous entities
    for i, (start, end) in enumerate(offsets):
        if start == end == 0:  # Skip special tokens ([CLS], [SEP])
            continue

        label = id2label[pred_ids[i]]

        if label == 'O':
            if curr_ent:
                entities.append(curr_ent)
                curr_ent = None
        else:
            tag, ent_type = label.split('-')
            if ent_type == 'PER': ent_type = 'PERSON'

            if tag == 'B':
                if curr_ent:
                    entities.append(curr_ent)
                curr_ent = {'type': ent_type, 'start': start, 'end': end}
            elif tag == 'I':
                if curr_ent and curr_ent['type'] == ent_type:
                    curr_ent['end'] = end
                else:
                    if curr_ent:
                        entities.append(curr_ent)
                    curr_ent = {'type': ent_type, 'start': start, 'end': end}

    if curr_ent:
        entities.append(curr_ent)

    # format into the expected string structure
    str_ents = []
    for ent in entities:
        ent_str = text[ent['start']:ent['end']]
        str_ents.append(f"{ent['type']}: {ent_str}")

    return " | ".join(str_ents)

def predict_llama_ner(text):
    """Return Llama JSON-like NER output as a string for one text."""
    sys_prompt = (
        "You are an expert Named Entity Recognition system. "
        "Extract named entities from the text. Find all PERSON, ORG (organizations), and LOC (locations). "
        "Return the output STRICTLY as a single string. "
        "Do not group entities. Repeat the tag for each distinct entity. "
        "Do not use JSON format. Do not add any conversational text. Don't add any note. If there are no entities for a tag then don't include it in the output. "
        "Remember to NOT ADD any notes or other text just the output in the correct format. "
    )

    messages = [
        {"role": "system", "content": sys_prompt},

        {"role": "user", "content": "Text: Barack Obama spoke at Stanford University in California.\nEntities:"},
        {"role": "assistant", "content": "PERSON: Barack Obama | ORG: Stanford University | LOC: California"},

        {"role": "user", "content": "Text: Angela Merkel met Emmanuel Macron in Paris.\nEntities:"},
        {"role": "assistant", "content": "PERSON: Angela Merkel | PERSON: Emmanuel Macron | LOC: Paris"},

        {"role": "user", "content": "Text: Apple Inc. announced a donation to UNICEF in New York.\nEntities:"},
        {"role": "assistant", "content": "ORG: Apple Inc. | ORG: UNICEF | LOC: New York"},

        {"role": "user", "content": f"Text: {text}\nEntities:"}
    ]

    processed_inputs = llama_tok.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True, return_dict=True)
    input_ids = processed_inputs['input_ids'].to(llama_model.device)
    attention_mask = processed_inputs['attention_mask'].to(llama_model.device)
    with torch.no_grad():
        gen = llama_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            temperature=0.01
        )
    output_ids = gen[0][input_ids.shape[1]:]
    output = llama_tok.decode(output_ids, skip_special_tokens=True)

    return output

# -----------------------------
# 10) Run predictions
# -----------------------------
rows = []
for i, sample in enumerate(wikiann_eval):
    text = sample['tokens']
    if isinstance(text, list):
        text = ' '.join(text)

    # TODO: Uncomment after implementing prediction functions.
    bert_pred = predict_bert_ner(text)
    llama_pred = predict_llama_ner(text)

    rows.append({
        'id': i,
        'bert_prediction': bert_pred,
        'llama_prediction': llama_pred,
    })

print('Prepared rows:', len(rows))
print(rows[0] if rows else 'No rows')

# -----------------------------
# 11) Evaluate predictions
# -----------------------------
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def extract_gold_bio_tags(wikiann_sample, label_name_map={'PER': 'PERSON', 'LOC': 'LOCATION', 'ORG': 'ORG'}):
    """Convert WikiANN BIO tags to BIO label sequences."""
    # WikiANN tag names are fixed: O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC
    tag_names_wikiann = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

    tokens = wikiann_sample['tokens']
    raw_ner_tags = wikiann_sample.get('ner_tags', [])

    bio_tags = []
    for tag_idx in raw_ner_tags:
        raw_label = tag_names_wikiann[tag_idx] if tag_idx < len(tag_names_wikiann) else 'O'
        if raw_label == 'O':
            bio_tags.append('O')
        else:
            prefix, base_label = raw_label.split('-', 1)
            mapped_label = f'{prefix}-{label_name_map.get(base_label, base_label)}'
            bio_tags.append(mapped_label)
    return bio_tags

def parse_prediction_string_to_bio(pred_string, tokens):
    """Parse prediction string (e.g., 'PERSON: name | ORG: org') to BIO tags."""
    # Simple fallback: if prediction is placeholder, return all 'O'
    # TODO: Parse actual prediction strings (students implement this)
    entities = {'PERSON': set(), 'ORG': set(), 'LOC': set()}
    if ':' in pred_string:
        parts = pred_string.split('|')
        for part in parts:
            if ':' in part:
                lbl, val = part.split(':', 1)
                lbl = lbl.strip().upper()
                if lbl == 'PER': lbl = 'PERSON'
                if lbl in entities:
                    entities[lbl].add(val.strip().lower())

    # reconstruct text and map token character spans
    text = " ".join(tokens)
    token_spans = []
    curr_idx = 0
    for t in tokens:
        start = text.find(t, curr_idx)
        end = start + len(t)
        token_spans.append((start, end))
        curr_idx = end

    tags = ['O'] * len(tokens)
    lower_text = text.lower()

    tag_map = {'PERSON': 'PERSON', 'ORG': 'ORG', 'LOC': 'LOCATION'}

    for label, bio_label in tag_map.items():
        # sort by length descending to match longest entity spans first
        for mention in sorted(entities.get(label, set()), key=len, reverse=True):
            if not mention: continue

            start_char = lower_text.find(mention)
            if start_char < 0: continue
            end_char = start_char + len(mention)

            # find tokens that overlap with this character span
            overlapping_indices = [
                i for i, (ts, te) in enumerate(token_spans)
                if not (te <= start_char or ts >= end_char)
            ]

            if overlapping_indices:
                tags[overlapping_indices[0]] = f"B-{bio_label}"
                for i in overlapping_indices[1:]:
                    tags[i] = f"I-{bio_label}"

    return tags

# Collect gold and predicted sequences
gold_sequences = []
bert_sequences = []
llama_sequences = []

for i, sample in enumerate(wikiann_eval):
    gold_bio = extract_gold_bio_tags(sample)
    gold_sequences.append(gold_bio)

    bert_sequences.append(parse_prediction_string_to_bio(rows[i]['bert_prediction'], sample['tokens']))
    llama_sequences.append(parse_prediction_string_to_bio(rows[i]['llama_prediction'], sample['tokens']))

# Compute metrics
bert_precision = precision_score(gold_sequences, bert_sequences, zero_division=0)
bert_recall = recall_score(gold_sequences, bert_sequences, zero_division=0)
bert_f1 = f1_score(gold_sequences, bert_sequences, zero_division=0)

llama_precision = precision_score(gold_sequences, llama_sequences, zero_division=0)
llama_recall = recall_score(gold_sequences, llama_sequences, zero_division=0)
llama_f1 = f1_score(gold_sequences, llama_sequences, zero_division=0)

print('\n=== BERT NER Metrics ===')
print(f'Precision: {bert_precision:.4f}')
print(f'Recall: {bert_recall:.4f}')
print(f'F1: {bert_f1:.4f}')
print(classification_report(gold_sequences, bert_sequences, zero_division=0))

print('\n=== Llama NER Metrics ===')
print(f'Precision: {llama_precision:.4f}')
print(f'Recall: {llama_recall:.4f}')
print(f'F1: {llama_f1:.4f}')
print(classification_report(gold_sequences, llama_sequences, zero_division=0))

# -----------------------------
# 6) Save CSV
# -----------------------------
with open(OUT_CSV, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'bert_prediction', 'llama_prediction'])
    writer.writeheader()
    writer.writerows(rows)

print('Saved:', OUT_CSV)

Loaded examples: 64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Prepared rows: 64
{'id': 0, 'bert_prediction': 'ORG: Sioux Falls Arena | LOC: Sioux Falls , South Dakota', 'llama_prediction': 'LOC: Sioux Falls | LOC: South Dakota'}

=== BERT NER Metrics ===
Precision: 0.8193
Recall: 0.8293
F1: 0.8242
              precision    recall  f1-score   support

    LOCATION       0.74      0.83      0.78        24
         ORG       0.81      0.83      0.82        30
      PERSON       0.92      0.82      0.87        28

   micro avg       0.82      0.83      0.82        82
   macro avg       0.82      0.83      0.82        82
weighted avg       0.83      0.83      0.83        82


=== Llama NER Metrics ===
Precision: 0.3900
Recall: 0.4756
F1: 0.4286
              precision    recall  f1-score   support

    LOCATION       0.24      0.46      0.32        24
         ORG       0.50      0.20      0.29        30
      PERSON       0.51      0.79      0.62        28

   micro avg       0.39      0.48      0.43        82
   macro avg       0.42      0.48      

# How to submit
1) generate the `.csv` file at `OUT_CSV` (must be called `exercise_predictions.csv`)
2) save the notebook as `.py` (any name is fine)
3) download both and put them in a new empty directory called `exercise_predictions`
4) zip the directory `exercise_predictions`
5) go to https://www.codabench.org/competitions/16004/, register to the task using your email @studenti.unipd.it
6) go to the submission page and create a new submission uploading `exercise_predictions.zip`